In [2]:
SHARD_DIR = "../data/0_raw/ensemble"
THRESHOLD = 0.8
URI = "neo4j+s://a49d25b9.databases.neo4j.io/browser/"
USER = "neo4j"
PASSWORD = "YVxwqHV1Qyr33y35HaFV_kVOuIw2qGysZ3JyI8WN37o"
DATABASE = "neo4j"
BATCH_SIZE = 10_000

import glob
shards = sorted(glob.glob(f"{SHARD_DIR}/predictions_*.parquet"))
print(f"{len(shards)} shards found")

222 shards found


In [3]:
import pandas as pd

frames = [pd.read_parquet(s, columns=["Drug_ID", "Target_ID", "ensemble_probability"]) for s in shards]
preds = pd.concat(frames, ignore_index=True)
preds = preds[preds["ensemble_probability"] > THRESHOLD]
print(f"{len(preds):,} predictions above {THRESHOLD} (out of {sum(len(f) for f in frames):,} rows)")
preds.head()

152,701,941 predictions above 0.8 (out of 442,284,138 rows)


,Drug_ID,Target_ID,ensemble_probability
3,NPASS_D9101,NPASS_T2250,0.876465
11,NPASS_D9107,NPASS_T2250,0.906250
14,NPASS_D9110,NPASS_T2250,0.903320
30,NPASS_D9126,NPASS_T2250,0.844238
34,NPASS_D2910,NPASS_T2250,0.894531


In [ ]:
# delete existing relationships, then insert the new ones
# pip install neo4j  (uncomment if needed)
from neo4j import GraphDatabase

rows = [{"substance": d, "target": t, "probability": float(p)}
        for d, t, p in zip(preds["Drug_ID"], preds["Target_ID"], preds["ensemble_probability"])]

with GraphDatabase.driver(URI, auth=(USER, PASSWORD)) as driver:
    driver.verify_connectivity()
    with driver.session(database=DATABASE) as session:
        summary = session.run(
            "MATCH (s:Substance)-[r:ACTIVITY_PREDICTION]-(t:Target) DELETE r"
        ).consume()
        print("deleted:", summary.counters.relationships_deleted, "relationships")

        inserted = 0
        for i in range(0, len(rows), BATCH_SIZE):
            batch = rows[i:i + BATCH_SIZE]
            session.run(
                "UNWIND $rows AS row "
                "MATCH (s:Substance {id: row.substance}), (t:Target {id: row.target}) "
                "CREATE (s)-[:ACTIVITY_PREDICTION {model: 'ensemble', probability: row.probability}]->(t)",
                rows=batch,
            ).consume()
            inserted += len(batch)
            print(f"inserted {inserted:,} / {len(rows):,}", end="\r")

print(f"\ndone: {inserted:,} ensemble ACTIVITY_PREDICTION relationships created")